In [1]:
!nvidia-smi
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

Sun Apr  5 12:33:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          Off |   00000000:04:00.0 Off |                    0 |
| N/A   35C    P0             72W /  700W |       0MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
import os
import time
import copy
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tqdm.auto import tqdm

In [3]:
SEED = 68
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

BATCH_SIZE = 32
NUM_EPOCHS = 10
LR = 1e-4
NUM_WORKERS = 2
IMAGE_SIZE = 224

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

SAVE_PATH = "best_shufflenet_v2_single_label.pth"

Using device: cuda


In [4]:
# Importing Datasets from Kaggle

from google.colab import userdata
!export KAGGLE_API_TOKEN={userdata.get('KAGGLE_API_TOKEN')}

import kagglehub

# Download latest version of datasets
ds2 = kagglehub.dataset_download("jeftaadriel/oia-odir-dataset")

100%|██████████| 1.59G/1.59G [01:43<00:00, 16.4MB/s]

Extracting files...


In [5]:
print(ds2)

/root/.cache/kagglehub/datasets/jeftaadriel/oia-odir-dataset/versions/1


In [6]:
label_cols = [
    "NORMAL",
    "DIABETIC_RETINOPATHY",
    "GLAUCOMA",
    "CATARACT",
    "AGE_RELATED_MACULAR_DEGENERATION",
    "HYPERTENSION",
    "MYOPIA",
    "OTHER_DISEASES",
]

class_names = label_cols.copy()
num_classes = len(class_names)

In [7]:
# Convert files to csv format
off_site_file = pd.read_excel(f"{ds2}/Off-site Test Set/Annotation/off-site test annotation (English).xlsx")
off_site_file.to_csv("off_site_test_annotation.csv", index=None, header=True)

on_site_file = pd.read_excel(f"{ds2}/On-site Test Set/Annotation/on-site test annotation (English).xlsx")
on_site_file.to_csv("on_site_test_annotation.csv", index=None, header=True)

training_file = pd.read_excel(f"{ds2}/Training Set/Annotation/training annotation (English).xlsx")
training_file.to_csv("training_annotation.csv", index=None, header=True)

# Base paths for images
base_path_off_site = ds2 + '/Off-site Test Set/Images/'
base_path_on_site = ds2 + '/On-site Test Set/Images/'
base_path_training = ds2 + '/Training Set/Images/'

# Add image paths to a DataFrame
def process_fundus_image_paths(df, base_path, left_col='Left-Fundus', right_col='Right-Fundus'):
    def add_image_path(filename, current_base_path):
        if pd.isna(filename):
            return filename
        return os.path.join(current_base_path, filename)

    df[left_col] = df[left_col].apply(lambda x: add_image_path(x, base_path))
    df[right_col] = df[right_col].apply(lambda x: add_image_path(x, base_path))
    return df

# Apply to each DataFrame
off_site_file = process_fundus_image_paths(off_site_file, base_path_off_site)
on_site_file = process_fundus_image_paths(on_site_file, base_path_on_site)
training_file = process_fundus_image_paths(training_file, base_path_training)

# Combine all DataFrames into one
df_ds2 = pd.concat([off_site_file, on_site_file, training_file], ignore_index=True)

# Remove specified columns
columns_to_remove = ['ID', 'Patient Age', 'Patient Sex', 'N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']
df_ds2 = df_ds2.drop(columns=columns_to_remove)

# Convert Keywords into diagnosis for each eye
disease_mapping = {
    'normal': 'NORMAL',
    'diabetic retinopathy': 'DIABETIC_RETINOPATHY',
    'glaucoma': 'GLAUCOMA',
    'cataract': 'CATARACT',
    'macular degeneration': 'AGE_RELATED_MACULAR_DEGENERATION',
    'age-related macular degeneration': 'AGE_RELATED_MACULAR_DEGENERATION',
    'hypertensive retinopathy': 'HYPERTENSION',
    'myopia': 'MYOPIA',
}

df_left = df_ds2[['Left-Fundus', 'Left-Diagnostic Keywords']].copy()
df_right = df_ds2[['Right-Fundus', 'Right-Diagnostic Keywords']].copy()

df_left.columns = ['image_path', 'eye_disease']
df_right.columns = ['image_path', 'eye_disease']

# Get unique disease codes for column names
disease_codes = list(set(disease_mapping.values()))
disease_codes.append('OTHER_DISEASES')

def parse_diagnosis(df):
    # Initialize all disease columns with 0
    for disease in disease_codes:
        df[disease] = 0

    # Parse each row's diagnostic keywords
    for idx, keywords in df['eye_disease'].items():
        if pd.isna(keywords):
            continue

        # Convert to lowercase for matching
        keywords_lower = str(keywords).lower()

        disease_found = False

        # Check for each disease keyword in the mapping
        for keyword, disease_code in disease_mapping.items():
            if keyword in keywords_lower:
                df.at[idx, disease_code] = 1
                disease_found = True

        if not disease_found:
            df.at[idx, 'OTHER_DISEASES'] = 1

    df = df.drop('eye_disease', axis=1)

    return df

# Apply to both dataframes
df_left = parse_diagnosis(df_left)
df_right = parse_diagnosis(df_right)

# Combine to final dataset
df_ds2 = pd.concat([df_left, df_right], ignore_index=True)

print(df_left)
print(df_right)

                                             image_path  CATARACT  \
0     /root/.cache/kagglehub/datasets/jeftaadriel/oi...         0   
1     /root/.cache/kagglehub/datasets/jeftaadriel/oi...         0   
2     /root/.cache/kagglehub/datasets/jeftaadriel/oi...         0   
3     /root/.cache/kagglehub/datasets/jeftaadriel/oi...         0   
4     /root/.cache/kagglehub/datasets/jeftaadriel/oi...         0   
...                                                 ...       ...   
4995  /root/.cache/kagglehub/datasets/jeftaadriel/oi...         0   
4996  /root/.cache/kagglehub/datasets/jeftaadriel/oi...         0   
4997  /root/.cache/kagglehub/datasets/jeftaadriel/oi...         0   
4998  /root/.cache/kagglehub/datasets/jeftaadriel/oi...         0   
4999  /root/.cache/kagglehub/datasets/jeftaadriel/oi...         0   

      AGE_RELATED_MACULAR_DEGENERATION  MYOPIA  HYPERTENSION  GLAUCOMA  \
0                                    0       0             1         0   
1                      

In [8]:
# build single-label dataframe
df_model = df_ds2.copy()

# Keep only needed columns
df_model = df_model[["image_path"] + label_cols].copy()

# Drop rows with missing image path
df_model = df_model.dropna(subset=["image_path"]).reset_index(drop=True)

# Keep only files that exist
df_model = df_model[df_model["image_path"].apply(os.path.exists)].reset_index(drop=True)

# Convert labels to numeric
df_model[label_cols] = df_model[label_cols].astype(int)

# Sum active labels per row
label_sum = df_model[label_cols].sum(axis=1)

# Drop rows with no active class
df_model = df_model[label_sum >= 1].reset_index(drop=True)

# Convert one-hot / multi-hot labels to a single target class index
# If multiple labels are active, argmax picks the first active column.
df_model["target"] = np.argmax(df_model[label_cols].values, axis=1)

print("Dataset size after filtering:", len(df_model))
print(df_model[["image_path", "target"]].head())

class_names = label_cols.copy()
num_classes = len(class_names)

Dataset size after filtering: 10000
                                          image_path  target
0  /root/.cache/kagglehub/datasets/jeftaadriel/oi...       5
1  /root/.cache/kagglehub/datasets/jeftaadriel/oi...       7
2  /root/.cache/kagglehub/datasets/jeftaadriel/oi...       7
3  /root/.cache/kagglehub/datasets/jeftaadriel/oi...       7
4  /root/.cache/kagglehub/datasets/jeftaadriel/oi...       7


In [9]:
# train/test split
train_df, test_df = train_test_split(
    df_model,
    test_size=0.2,
    random_state=SEED,
    stratify=df_model["target"],
    shuffle=True,
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train size:", len(train_df))
print("Test size:", len(test_df))

Train size: 8000
Test size: 2000


In [10]:
def crop_black_borders(image):
    img_np = np.array(image)
    black_threshold = 10

    non_black_pixels = np.any(img_np > black_threshold, axis=2)
    rows = np.any(non_black_pixels, axis=1)
    cols = np.any(non_black_pixels, axis=0)

    if not rows.any() or not cols.any():
        return image

    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]

    cropped_img_np = img_np[rmin:rmax + 1, cmin:cmax + 1]
    return Image.fromarray(cropped_img_np)

In [11]:
# transformation
weights = models.ShuffleNet_V2_X1_0_Weights.DEFAULT

train_transform = transforms.Compose([
    transforms.Lambda(crop_black_borders),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

test_transform = transforms.Compose([
    transforms.Lambda(crop_black_borders),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

In [12]:
# dataset class
class ODIRSingleLabelDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")
        label = torch.tensor(int(row["target"]), dtype=torch.long)

        if self.transform:
            image = self.transform(image)

        return image, label

In [13]:
# dataset/loader
train_dataset = ODIRSingleLabelDataset(train_df, transform=train_transform)
test_dataset = ODIRSingleLabelDataset(test_df, transform=test_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

In [14]:
sample_img, sample_label = train_dataset[0]
print("Single image shape:", sample_img.shape)
print("Single label:", sample_label)
print("Single label dtype:", sample_label.dtype)

images, labels = next(iter(train_loader))
print("Batch image shape:", images.shape)
print("Batch label shape:", labels.shape)
print("Batch label dtype:", labels.dtype)

Single image shape: torch.Size([3, 224, 224])
Single label: tensor(7)
Single label dtype: torch.int64
Batch image shape: torch.Size([32, 3, 224, 224])
Batch label shape: torch.Size([32])
Batch label dtype: torch.int64


In [15]:
# building the model
model = models.shufflenet_v2_x1_0(weights=weights)
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_classes)
model = model.to(device)

print("Output classes:", model.fc.out_features)

Downloading: "https://download.pytorch.org/models/shufflenetv2_x1-5666bf0f80.pth" to /root/.cache/torch/hub/checkpoints/shufflenetv2_x1-5666bf0f80.pth


100%|██████████| 8.79M/8.79M [00:00<00:00, 236MB/s]

Output classes: 8


In [16]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

In [17]:
def train_one_epoch(model, loader, criterion, optimizer, device, epoch_num=None, num_epochs=None):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    progress_bar = tqdm(
        loader,
        desc=f"Epoch {epoch_num}/{num_epochs} [Train]" if epoch_num is not None else "Training",
        leave=False
    )

    for batch_idx, (images, labels) in enumerate(progress_bar, start=1):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        avg_loss = running_loss / total
        avg_acc = correct / total

        progress_bar.set_postfix(
            batch_loss=f"{loss.item():.4f}",
            avg_loss=f"{avg_loss:.4f}",
            avg_acc=f"{avg_acc:.4f}",
        )

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = correct / len(loader.dataset)
    return epoch_loss, epoch_acc

In [18]:
def train_one_epoch(model, loader, criterion, optimizer, device, epoch_num=None, num_epochs=None):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    progress_bar = tqdm(
        loader,
        desc=f"Epoch {epoch_num}/{num_epochs} [Train]" if epoch_num is not None else "Training",
        leave=False
    )

    for batch_idx, (images, labels) in enumerate(progress_bar, start=1):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        avg_loss = running_loss / total
        avg_acc = correct / total

        progress_bar.set_postfix(
            batch_loss=f"{loss.item():.4f}",
            avg_loss=f"{avg_loss:.4f}",
            avg_acc=f"{avg_acc:.4f}",
        )

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = correct / len(loader.dataset)
    return epoch_loss, epoch_acc

In [19]:
# training the model
def evaluate(model, loader, criterion, device, epoch_num=None, num_epochs=None):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    all_probs = []
    all_preds = []
    all_labels = []

    progress_bar = tqdm(
        loader,
        desc=f"Epoch {epoch_num}/{num_epochs} [Test]" if epoch_num is not None else "Evaluating",
        leave=False
    )

    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(progress_bar, start=1):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)

            running_loss += loss.item() * images.size(0)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            avg_loss = running_loss / total
            avg_acc = correct / total

            progress_bar.set_postfix(
                batch_loss=f"{loss.item():.4f}",
                avg_loss=f"{avg_loss:.4f}",
                avg_acc=f"{avg_acc:.4f}",
            )

            all_probs.append(probs.cpu())
            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())

    all_probs = torch.cat(all_probs, dim=0)
    all_preds = torch.cat(all_preds, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = correct / len(loader.dataset)

    return epoch_loss, epoch_acc, all_probs, all_preds, all_labels

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [21]:
best_test_loss = float("inf")
best_model_wts = copy.deepcopy(model.state_dict())

for epoch in range(NUM_EPOCHS):
    start_time = time.time()

    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device,
        epoch_num=epoch + 1, num_epochs=NUM_EPOCHS
    )

    test_loss, test_acc, test_probs, test_preds, test_labels = evaluate(
        model, test_loader, criterion, device,
        epoch_num=epoch + 1, num_epochs=NUM_EPOCHS
    )

    scheduler.step()

    if test_loss < best_test_loss:
        best_test_loss = test_loss
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), SAVE_PATH)

    epoch_time = time.time() - start_time

    print(f"Epoch {epoch + 1}/{NUM_EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Test  Loss: {test_loss:.4f} | Test  Acc: {test_acc:.4f}")
    print(f"Time: {epoch_time:.1f}s")
    print("-" * 50)

model.load_state_dict(best_model_wts)
print(f"Best model loaded from: {SAVE_PATH}")

Epoch 1/10 [Train]:   0%|          | 0/250 [00:00<?, ?it/s]

Epoch 1/10 [Test]:   0%|          | 0/63 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7fdef81a4680>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process


Epoch 1/10
Train Loss: 1.6370 | Train Acc: 0.4590
Test  Loss: 1.3266 | Test  Acc: 0.4950
Time: 660.7s
--------------------------------------------------


Epoch 2/10 [Train]:   0%|          | 0/250 [00:00<?, ?it/s]

...

Epoch 9/10 [Test]:   0%|          | 0/63 [00:00<?, ?it/s]

Epoch 9/10
Train Loss: 0.5851 | Train Acc: 0.7939
Test  Loss: 1.0581 | Test  Acc: 0.6070
Time: 698.2s
--------------------------------------------------


Epoch 10/10 [Train]:   0%|          | 0/250 [00:00<?, ?it/s]

Epoch 10/10 [Test]:   0%|          | 0/63 [00:00<?, ?it/s]

Epoch 10/10
Train Loss: 0.5594 | Train Acc: 0.8081
Test  Loss: 1.0778 | Test  Acc: 0.6040
Time: 689.1s
--------------------------------------------------
Best model loaded from: best_shufflenet_v2_single_label.pth


In [22]:
import os
print(os.getcwd())
print(os.listdir())

/content
['.config', 'best_shufflenet_v2_single_label.pth', 'off_site_test_annotation.csv', 'drive', 'training_annotation.csv', 'on_site_test_annotation.csv', 'sample_data']


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CHECKPOINT_PATH = "/content/SECOND_MODEL_best_shufflenet_v2_single_label.pth"

state_dict = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

num_classes = 8

model = models.shufflenet_v2_x1_0(weights=None)
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_classes)

model.load_state_dict(state_dict)
model = model.to(DEVICE)
model.eval()

In [23]:
import torch
import os

os.makedirs("/content/drive/MyDrive/models", exist_ok=True)

save_path = "/content/drive/MyDrive/models/SECOND_MODEL_best_shufflenet_v2_single_label.pth"
torch.save(model.state_dict(), save_path)

print("Saved at:", save_path)

Saved at: /content/drive/MyDrive/models/SECOND_MODEL_best_shufflenet_v2_single_label.pth


In [24]:
final_test_loss, final_test_acc, final_test_probs, final_test_preds, final_test_labels = evaluate(
    model, test_loader, criterion, device
)

print("Final Test Loss:", round(final_test_loss, 4))
print("Final Test Accuracy:", round(final_test_acc, 4))

y_true = final_test_labels.numpy()
y_pred = final_test_preds.numpy()

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

Evaluating:   0%|          | 0/63 [00:00<?, ?it/s]

Final Test Loss: 0.9982
Final Test Accuracy: 0.62

Classification Report:
                                  precision    recall  f1-score   support

                          NORMAL     0.6552    0.7213    0.6867       872
            DIABETIC_RETINOPATHY     0.0000    0.0000    0.0000        27
                        GLAUCOMA     0.5821    0.4432    0.5032        88
                        CATARACT     0.8000    0.8352    0.8172        91
AGE_RELATED_MACULAR_DEGENERATION     0.0000    0.0000    0.0000        73
                    HYPERTENSION     0.0000    0.0000    0.0000        46
                          MYOPIA     0.7500    0.7714    0.7606        70
                  OTHER_DISEASES     0.5484    0.6030    0.5744       733

                        accuracy                         0.6200      2000
                       macro avg     0.4170    0.4218    0.4178      2000
                    weighted avg     0.5749    0.6200    0.5959      2000



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
